# 🔄 Convertisseur .cha → JSONL

**Rôle :** Transformer des fichiers CHILDES `.cha` + audio en paires `{audio_path, text}` au format JSONL,  
prêtes à être consommées par le pipeline `04_whisper_preprocessing_augmentation_pipeline.ipynb`.

```
INPUT
  ├── data/cha/   → fichiers .cha  (transcriptions CHILDES avec timestamps %wor)
  └── data/audio/ → fichiers audio (.wav / .mp3 / .m4a ...)

OUTPUT
  └── input_pairs.jsonl  → {"id": ..., "audio_path": ..., "text": ..., "speaker": ...}
```

**Ce notebook fait uniquement :**
1. Parser les `.cha` et extraire les segments `%wor` avec timestamps
2. Matcher `.cha` ↔ audio par chemin relatif
3. Découper l'audio selon les timestamps (ffmpeg)
4. Exporter un JSONL propre

---
## Zone 0 : Dépendances & Configuration

In [1]:
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q tqdm pandas
print('✅ Prêt')

✅ Prêt


In [2]:
import re
import json
import subprocess
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import pandas as pd
from tqdm import tqdm

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Adapter selon ton environnement
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── Entrée ───────────────────────────────────────────────────────────────
    "cha_dir"          : Path("/content/drive/MyDrive/asr/data/cha"),
    "audio_dir"        : Path("/content/drive/MyDrive/asr/data/songs"),
    "audio_extensions" : [".wav", ".mp3", ".m4a", ".flac"],
    "max_segments": 500 ,  # None = tous les segments,

    # ── Sortie ───────────────────────────────────────────────────────────────
    # Dossier où les segments audio découpés seront sauvegardés
    "segments_dir"     : Path("/content/drive/MyDrive/asr/output/cha_segments"),
    # Fichier JSONL final — c'est LUI que tu passes au pipeline 04
    "output_jsonl"     : Path("/content/drive/MyDrive/asr/output/input_pairs.jsonl"),

    # ── Audio ────────────────────────────────────────────────────────────────
    "sample_rate"      : 16000,

    # ── Filtres pré-export (optionnels, légers) ──────────────────────────────
    # Durée minimale d'un segment pour être inclus (en ms)
    "min_duration_ms"  : 500,
    # Durée maximale (en ms) — les segments trop longs seront ignorés
    "max_duration_ms"  : 30000,
    # Nombre minimum de mots par segment
    "min_words"        : 1,

    # ── Filtrage par speaker ─────────────────────────────────────────────────
    # None = tous les speakers
    # Exemple : {"Target_Child", "Child", "KAT", "WIL"}
    "target_speakers"  : None,
}

CONFIG["segments_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["output_jsonl"].parent.mkdir(parents=True, exist_ok=True)

print('✅ Configuration chargée')
print(f'   .cha dir   : {CONFIG["cha_dir"]}')
print(f'   audio dir  : {CONFIG["audio_dir"]}')
print(f'   output     : {CONFIG["output_jsonl"]}')

✅ Configuration chargée
   .cha dir   : /content/drive/MyDrive/asr/data/cha
   audio dir  : /content/drive/MyDrive/asr/data/songs
   output     : /content/drive/MyDrive/asr/output/input_pairs.jsonl


---
## Zone 1 : Parsing des fichiers .cha

Extrait les segments `%wor` (word tier avec timestamps) de chaque fichier CHILDES.

In [5]:
@dataclass
class ChaSegment:
    """
    Représente un tour de parole extrait d'un fichier .cha.

    Attributs :
      speaker   : identifiant du locuteur (ex: 'KAT', 'Target_Child')
      text      : transcription nettoyée
      words     : liste de (mot, start_ms, end_ms)
      file_name : stem du fichier .cha source (ex: '01-1a')
    """
    speaker  : str
    text     : str
    words    : List[Tuple[str, int, int]]  # (word, start_ms, end_ms)
    file_name: str

    @property
    def start_ms(self) -> int:
        return self.words[0][1]

    @property
    def end_ms(self) -> int:
        return self.words[-1][2]

    @property
    def duration_ms(self) -> int:
        return self.end_ms - self.start_ms

    @property
    def num_words(self) -> int:
        return len(self.words)

In [6]:
# Tokens CHILDES à ignorer dans la transcription
_PUNCT_TOKENS  = {'?', '.', ',', '!', '+...', '0', 'xxx', 'yyy', 'www'}
_CHA_ANNO_RE   = re.compile(r'[&@\[\]<>]')   # Annotations CHAT
_TIMESTAMP_RE  = re.compile(r'^\d{4,}_\d{4,}$')


def _clean_word(token: str) -> Optional[str]:
    """
    Nettoie un token CHILDES.
    Retourne None si le token doit être ignoré.
    """
    if token in _PUNCT_TOKENS:
        return None
    # Supprimer les annotations CHAT (@, &, [], <>)
    cleaned = _CHA_ANNO_RE.sub('', token).strip()
    # Supprimer les marqueurs de troncature (ex: ba- )
    cleaned = re.sub(r'-+$', '', cleaned).strip()
    if not cleaned or len(cleaned) < 1:
        return None
    return cleaned


def parse_cha_file(cha_path: Path) -> List[ChaSegment]:
    """
    Parse un fichier .cha CHILDES et retourne la liste des ChaSegment.

    Lit les lignes %wor (word tier) et associe chaque mot à son timestamp.
    Format timestamp dans %wor : mot_XXXXX_YYYYY  (start_end en ms)
    """
    segments      = []
    current_spk   = None
    file_name     = cha_path.stem

    try:
        lines = cha_path.read_text(encoding='utf-8', errors='replace').splitlines()
    except Exception as e:
        print(f'  ⚠️  Impossible de lire {cha_path.name}: {e}')
        return []

    for line in lines:
        line = line.rstrip()

        # ── Tour principal : *SPEAKER: texte
        if line.startswith('*'):
            parts = line.split(':', 1)
            if parts:
                current_spk = parts[0].replace('*', '').strip()
            continue

        # ── Word tier : %wor: mot1_T1_T2 mot2_T3_T4 ...
        if line.startswith('%wor:'):
            if not current_spk:
                continue

            content = line.split(':', 1)[1].strip()
            # Supprimer les caractères de contrôle CLAN (\x15, \x01, etc.)
            content = re.sub(r'[\x00-\x1f]', ' ', content)

            tokens = content.split()
            words  = []
            i = 0

            while i < len(tokens):
                token = tokens[i]

                # Cas 1 : timestamp isolé (XXXXX_YYYYY) → attacher au mot précédent
                if _TIMESTAMP_RE.match(token):
                    if words:
                        m = re.match(r'(\d+)_(\d+)', token)
                        if m:
                            w = words[-1][0]
                            words[-1] = (w, int(m.group(1)), int(m.group(2)))
                    i += 1
                    continue

                # Cas 2 : mot avec timestamp intégré (mot_XXXXX_YYYYY)
                m_inline = re.match(r'^(.+?)_(\d{4,})_(\d{4,})$', token)
                if m_inline:
                    w = _clean_word(m_inline.group(1))
                    if w:
                        words.append((w, int(m_inline.group(2)), int(m_inline.group(3))))
                    i += 1
                    continue

                # Cas 3 : mot seul (timestamp à venir)
                w = _clean_word(token)
                if w:
                    words.append((w, None, None))
                i += 1

            # Garder uniquement les mots avec timestamp complet
            words_ts = [(w, s, e) for w, s, e in words
                        if s is not None and e is not None]

            if not words_ts:
                continue

            text = ' '.join(w for w, _, _ in words_ts)
            if not text.strip():
                continue

            segments.append(ChaSegment(
                speaker=current_spk,
                text=text,
                words=words_ts,
                file_name=file_name
            ))

    return segments


def parse_cha_directory(cha_dir: Path) -> List[ChaSegment]:
    """Parse tous les .cha d'un dossier (récursivement)."""
    cha_files = sorted(cha_dir.rglob('*.cha'))
    if not cha_files:
        print(f'⚠️  Aucun .cha trouvé dans {cha_dir}')
        return []

    print(f'   Fichiers .cha trouvés : {len(cha_files)}')
    all_segs = []
    for f in tqdm(cha_files, desc='Parsing .cha'):
        segs = parse_cha_file(f)
        all_segs.extend(segs)

    return all_segs


print('✅ Parseur .cha défini')

✅ Parseur .cha défini


In [7]:
print('=' * 70)
print('ZONE 1 : PARSING .CHA')
print('=' * 70)

all_segments = parse_cha_directory(CONFIG['cha_dir'])

# ── Statistiques ──────────────────────────────────────────────────────────
from collections import Counter

speakers   = Counter(s.speaker   for s in all_segments)
files      = Counter(s.file_name for s in all_segments)
durations  = [s.duration_ms / 1000 for s in all_segments]

print(f'\n   Total segments extraits : {len(all_segments)}')
print(f'   Fichiers .cha parsés    : {len(files)}')
print(f'   Speakers uniques        : {len(speakers)}')
print(f'   Durée totale audio      : {sum(durations)/60:.1f} min')
print(f'   Durée moy. par segment  : {sum(durations)/max(len(durations),1):.2f}s')
print(f'\n   Segments par speaker :')
for spk, count in speakers.most_common():
    total_s = sum(s.duration_ms for s in all_segments if s.speaker == spk) / 1000
    print(f'      {spk:<25} {count:>5} segments | {total_s:>7.1f}s')

ZONE 1 : PARSING .CHA
   Fichiers .cha trouvés : 247


Parsing .cha: 100%|██████████| 247/247 [00:26<00:00,  9.20it/s]



   Total segments extraits : 29763
   Fichiers .cha parsés    : 214
   Speakers uniques        : 28
   Durée totale audio      : 1242.4 min
   Durée moy. par segment  : 2.50s

   Segments par speaker :
      KAT                       14234 segments | 30709.5s
      LUS                        1890 segments |  4390.7s
      WIL                        1430 segments |  4049.9s
      MAT                        1413 segments |  4781.7s
      MAI                        1311 segments |  3908.0s
      NIN                        1217 segments |  3542.2s
      DYL                         972 segments |  2388.1s
      VIC                         894 segments |  2683.2s
      LSN                         885 segments |  2128.1s
      MAS                         865 segments |  3035.6s
      LAN                         724 segments |  2057.1s
      ENZ                         686 segments |  1393.5s
      ELI                         627 segments |  1523.8s
      RIT                         379 segme

---
## Zone 2 : Matching .cha ↔ Audio

In [8]:
def match_audio_files(cha_dir: Path, audio_dir: Path,
                      extensions: List[str]) -> Dict[str, Path]:
    """
    Construit un dictionnaire stem → audio_path en matchant
    les fichiers audio par chemin relatif (même structure de sous-dossiers).

    Exemple : cha/1/01-1a.cha  ↔  audio/1/01-1a.wav
    """
    audio_map = {}
    for ext in extensions:
        for audio_path in audio_dir.rglob(f'*{ext}'):
            # Clé = chemin relatif sans extension
            rel = str(audio_path.relative_to(audio_dir).with_suffix(''))
            audio_map[rel] = audio_path
            # Aussi indexer par stem seul (fallback)
            audio_map[audio_path.stem] = audio_path

    return audio_map


print('=' * 70)
print('ZONE 2 : MATCHING .CHA ↔ AUDIO')
print('=' * 70)

audio_map = match_audio_files(
    CONFIG['cha_dir'], CONFIG['audio_dir'], CONFIG['audio_extensions']
)

# Vérifier la couverture
cha_stems = set(s.file_name for s in all_segments)
matched   = cha_stems & set(audio_map.keys())
missing   = cha_stems - set(audio_map.keys())

print(f'\n   Fichiers .cha uniques   : {len(cha_stems)}')
print(f'   Fichiers audio indexés  : {len(set(audio_map.values()))}')
print(f'   ✅ Matchés               : {len(matched)}')
if missing:
    print(f'   ⚠️  Sans audio ({len(missing)})   : {sorted(missing)[:10]}')

ZONE 2 : MATCHING .CHA ↔ AUDIO

   Fichiers .cha uniques   : 214
   Fichiers audio indexés  : 245
   ✅ Matchés               : 214


---
## Zone 3 : Découpage audio + Filtres pré-export

In [17]:
def extract_segment_ffmpeg(audio_path: Path, start_ms: int, end_ms: int,
                            out_path: Path, sr: int = 16000) -> bool:
    """Extrait un segment audio via ffmpeg. Retourne True si succès."""
    if out_path.exists():
        return True
    cmd = [
        'ffmpeg',
        '-i',  str(audio_path),
        '-ss', str(start_ms / 1000.0),
        '-t',  str((end_ms - start_ms) / 1000.0),
        '-ar', str(sr),
        '-ac', '1',
        '-acodec', 'pcm_s16le',
        '-y',  str(out_path)
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=15)
        return True
    except Exception:
        return False


def apply_prefilters(seg: ChaSegment, cfg: Dict) -> Optional[str]:
    """
    Filtres légers pré-export (sans analyser l'audio).
    Retourne la raison de rejet, ou None si le segment passe.
    """
    if seg.duration_ms < cfg['min_duration_ms']:
        return f'too_short ({seg.duration_ms}ms)'
    if seg.duration_ms > cfg['max_duration_ms']:
        return f'too_long ({seg.duration_ms}ms)'
    if seg.num_words < cfg['min_words']:
        return f'too_few_words ({seg.num_words})'
    if cfg['target_speakers'] and seg.speaker not in cfg['target_speakers']:
        return f'speaker_excluded ({seg.speaker})'
    return None


def process_segments(segments: List[ChaSegment], audio_map: Dict[str, Path],
                     cfg: Dict) -> Tuple[List[Dict], pd.DataFrame]:
    """
    Pour chaque segment :
      1. Vérifie les filtres légers
      2. Trouve le fichier audio correspondant (.cha ↔ audio)
      3. Découpe le segment avec ffmpeg
      4. Produit un dict prêt pour le JSONL
      5. S'arrête dès que max_segments segments valides sont extraits
    """
    print('=' * 70)
    print('ZONE 3 : DÉCOUPAGE AUDIO')
    print('=' * 70)

    out_dir      = cfg['segments_dir']
    sr           = cfg['sample_rate']
    max_segments = cfg.get('max_segments', None)  # None = pas de limite
    pairs        = []
    report       = []
    stats        = {'ok': 0, 'no_audio': 0, 'prefilter': 0, 'ffmpeg_fail': 0}

    if max_segments is not None:
        print(f'\n   Limite fixée à {max_segments} segments valides')
    else:
        print(f'\n   Pas de limite (tous les segments seront traités)')

    for i, seg in enumerate(tqdm(segments, desc='Découpage')):

        # ── Arrêt dès que la limite de segments valides est atteinte ─────
        if max_segments is not None and stats['ok'] >= max_segments:
            print(f'\n   🛑 Limite de {max_segments} segments atteinte — arrêt.')
            break

        # ── Filtres pré-export ────────────────────────────────────────────
        reject = apply_prefilters(seg, cfg)
        if reject:
            stats['prefilter'] += 1
            report.append({'file': seg.file_name, 'speaker': seg.speaker,
                           'status': 'prefilter', 'reason': reject})
            continue

        # ── Trouver l'audio source (.cha ↔ audio) ────────────────────────
        # Cherche d'abord par chemin relatif (ex: '1/01-1a'),
        # puis par stem seul (ex: '01-1a') en fallback
        audio_path = audio_map.get(seg.file_name)
        if audio_path is None:
            stats['no_audio'] += 1
            report.append({'file': seg.file_name, 'speaker': seg.speaker,
                           'status': 'no_audio', 'reason': 'no matching audio'})
            continue

        # ── Créer le sous-dossier par speaker ────────────────────────────
        spk_dir = out_dir / seg.speaker
        spk_dir.mkdir(exist_ok=True)

        seg_id   = f"{seg.file_name}_{seg.speaker}_{i:05d}"
        out_path = spk_dir / f"{seg_id}.wav"

        # ── Découpe ffmpeg ────────────────────────────────────────────────
        ok = extract_segment_ffmpeg(
            audio_path, seg.start_ms, seg.end_ms, out_path, sr=sr
        )
        if not ok:
            stats['ffmpeg_fail'] += 1
            report.append({'file': seg.file_name, 'speaker': seg.speaker,
                           'status': 'ffmpeg_fail', 'reason': 'extraction failed'})
            continue

        # ── Préparer la paire pour le JSONL ──────────────────────────────
        pairs.append({
            'id'          : seg_id,
            'audio_path'  : str(out_path),
            'text'        : seg.text,
            'speaker'     : seg.speaker,
            'source_file' : seg.file_name,   # correspondance .cha source
            'source_audio': str(audio_path), # correspondance audio source
            'start_ms'    : seg.start_ms,
            'end_ms'      : seg.end_ms,
            'duration_sec': seg.duration_ms / 1000.0,
            'num_words'   : seg.num_words,
        })
        stats['ok'] += 1
        report.append({'file': seg.file_name, 'speaker': seg.speaker,
                       'status': 'ok', 'reason': None})

    total = len(segments)
    print(f'\n   Segments en entrée      : {total}')
    print(f'   ✅ Extraits avec succès  : {stats["ok"]}')
    print(f'   ⏭️  Filtrés (pré-export)  : {stats["prefilter"]}')
    print(f'   ⚠️  Audio manquant        : {stats["no_audio"]}')
    print(f'   ❌ Erreur ffmpeg          : {stats["ffmpeg_fail"]}')
    if max_segments is not None and stats['ok'] < max_segments:
        print(f'   ℹ️  Segments dispo < limite ({stats["ok"]} / {max_segments})')

    return pairs, pd.DataFrame(report)


pairs, process_report_df = process_segments(all_segments, audio_map, CONFIG)

ZONE 3 : DÉCOUPAGE AUDIO

   Limite fixée à 500 segments valides


Découpage:   2%|▏         | 506/29763 [12:54<12:26:04,  1.53s/it]


   🛑 Limite de 500 segments atteinte — arrêt.

   Segments en entrée      : 29763
   ✅ Extraits avec succès  : 500
   ⏭️  Filtrés (pré-export)  : 6
   ⚠️  Audio manquant        : 0
   ❌ Erreur ffmpeg          : 0


---
## Zone 4 : Export JSONL

In [18]:
def export_jsonl(pairs: List[Dict], out_path: Path) -> None:
    """Exporte les paires au format JSONL."""
    with open(out_path, 'w', encoding='utf-8') as f:
        for pair in pairs:
            f.write(json.dumps(pair, ensure_ascii=False) + '\n')
    print(f'✅ JSONL exporté : {out_path}')
    print(f'   {len(pairs)} paires écrites')


print('=' * 70)
print('ZONE 4 : EXPORT JSONL')
print('=' * 70)

export_jsonl(pairs, CONFIG['output_jsonl'])

# Export du rapport de traitement
report_path = CONFIG['output_jsonl'].parent / 'cha_conversion_report.csv'
process_report_df.to_csv(report_path, index=False)
print(f'   Rapport CSV    : {report_path}')

ZONE 4 : EXPORT JSONL
✅ JSONL exporté : /content/drive/MyDrive/asr/output/input_pairs.jsonl
   500 paires écrites
   Rapport CSV    : /content/drive/MyDrive/asr/output/cha_conversion_report.csv
